In [64]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord, Galactocentric
from matplotlib.lines import Line2D
import galpy.orbit as go
from galpy.potential import MWPotential2014

In [57]:
cs = pd.read_csv('../data/true_sources_basta_output_final.csv',dtype={'source_id':str}) #stars with distances from basta
unconfirmed = pd.read_csv('../data/interp_d_unconfirmed_stars.csv',dtype={'source_id':str}) #stars which aren't confirmed to be RGs (pink diamonds in galaxia plot)

both = pd.concat([cs,unconfirmed])

In [58]:
#query ra, dec, and velocities from gaia
source_ids = both['source_id'].tolist()

# Process in batches of 1000
batch_size = 1000
all_results = []

for i in range(0, len(source_ids), batch_size):
    batch = source_ids[i:i+batch_size]
    id_string = ','.join(map(str, batch))
    
    query = f"""
        SELECT source_id, ra, dec,
               pmra, pmdec, radial_velocity
        FROM gaiadr3.gaia_source 
        WHERE source_id IN ({id_string})
    """
    
    job = Gaia.launch_job(query)
    all_results.append(job.get_results().to_pandas())

results_df = pd.concat(all_results, ignore_index=True)
results_df['source_id'] = results_df['source_id'].astype(str)

cs = cs.merge(results_df,on='source_id',how='left')
unconfirmed = unconfirmed.merge(results_df,on='source_id',how='left')

In [59]:
#define galactocentric frame
gc_frame = Galactocentric(
    galcen_distance=8.2 * u.kpc,
    z_sun=20.8 * u.pc,
    galcen_v_sun=[11.1, 244.0, 7.25] * u.km/u.s
)

In [60]:
#convert to xyz coordinates (cs)
cs_coords = SkyCoord(
    ra=cs['ra'].values * u.deg,
    dec=cs['dec'].values * u.deg,
    distance=cs['distance'].values * u.pc,
    frame='icrs'
)

gc_coords = cs_coords.transform_to(gc_frame)

cs['X'] = gc_coords.x.to(u.kpc).value
cs['Y'] = gc_coords.y.to(u.kpc).value
cs['Z'] = gc_coords.z.to(u.kpc).value

cs.to_csv('../data/true_sources_final_XYZ.csv',index=False)

In [61]:
#convert to xyz coordinates (unconfirmed sample)
unc_coords = SkyCoord(
    ra=unconfirmed['ra'].values * u.deg,
    dec = unconfirmed['dec'].values * u.deg,
    distance = unconfirmed['dist_kpc_est'].values * u.kpc,
    frame = 'icrs'
)

gc_coords = unc_coords.transform_to(gc_frame)

unconfirmed['X'] = gc_coords.x.to(u.kpc).value
unconfirmed['Y'] = gc_coords.y.to(u.kpc).value
unconfirmed['Z'] = gc_coords.z.to(u.kpc).value

unconfirmed.to_csv('../data/unconfirmed_XYZ.csv',index=False)

In [63]:
#ok now kinematics for stars with radial velocities
cs = cs.dropna(subset=['radial_velocity', 'pmra', 'pmdec', 'ra', 'dec']).copy() #must have all of these quantities
print(len(cs))

1721


In [65]:
#make skycoord object
coords = SkyCoord(
    ra=cs['ra'].values * u.deg,
    dec=cs['dec'].values * u.deg,
    distance=cs['distance'].values * u.pc,
    pm_ra_cosdec=cs['pmra'].values * u.mas/u.yr,
    pm_dec=cs['pmdec'].values * u.mas/u.yr,
    radial_velocity=cs['radial_velocity'].values * u.km/u.s,
    frame='icrs'
)


#transfrom to galactocentric frame
gc_coords = coords.transform_to(gc_frame)


#xyz positions
X = gc_coords.x.to(u.kpc).value
Y = gc_coords.y.to(u.kpc).value
Z = gc_coords.z.to(u.kpc).value

#xyz velocities
vx = gc_coords.v_x.to(u.km/u.s).value
vy = gc_coords.v_y.to(u.km/u.s).value
vz = gc_coords.v_z.to(u.km/u.s).value


#cylindrical coordinates
R = np.sqrt(X**2 + Y**2)
phi = np.arctan2(Y, X)

#cylindrical velocity components
V_R = vx * np.cos(phi) + vy * np.sin(phi)
V_phi = vx * np.sin(phi) - vy * np.cos(phi)
V_Z = vz

#toomre diagram velocities
V_LSR = 232.0  # km/s
V_phi_rel = V_phi - V_LSR
V_perp = np.sqrt(V_R**2 + V_Z**2)

# Total peculiar velocity
v_tot = np.sqrt(V_R**2 + V_phi_rel**2 + V_Z**2)

#solar values
RO = 8.2       # Galactocentric distance of the Sun [kpc]
VO = 232.0     # Circular velocity / velocity scale [km/s]

pot = MWPotential2014


# Integration times: 10 Gyr with 1 Myr spacing
ts = np.linspace(0, 10_000, 10_001) * u.Myr

# Output arrays
Z_max = np.full(len(cs), np.nan)
eccentricity = np.full(len(cs), np.nan)
Energy = np.full(len(cs), np.nan)
Lz = np.full(len(cs), np.nan)
energy_drift_pct = np.full(len(cs), np.nan)

#integrate each orbit
for i in range(len(cs)):

    orb = go.Orbit(
        [
            R[i] * u.kpc,
            V_R[i] * u.km / u.s,
            V_phi[i] * u.km / u.s,
            Z[i] * u.kpc,
            V_Z[i] * u.km / u.s,
            phi[i] * u.rad
        ],
        ro=RO * u.kpc,
        vo=VO * u.km / u.s
    )

    # High-accuracy integration method
    orb.integrate(ts, pot, method='dopr54_c')

    # Energy along the orbit
    energy = np.asarray(
        orb.E(ts, pot=pot, use_physical=True)
    )

    E0 = float(energy[0])
    Eend = float(energy[-1])

    Energy[i] = E0
    energy_drift_pct[i] = 100.0 * (Eend - E0) / abs(E0)

    # Orbital eccentricity
    eccentricity[i] = float(orb.e(pot=pot))

    # Angular momentum
    Lz[i] = float(orb.Lz(use_physical=True))

    # Maximum vertical excursion
    z_values = np.asarray(
        orb.z(ts, use_physical=True)
    )
    Z_max[i] = np.max(np.abs(z_values))

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(cs)} stars")

#save to dataframe
cs['Z_max'] = Z_max
cs['X'] = X
cs['Y'] = Y
cs['Z'] = Z
cs['vx'] = vx
cs['vy'] = vy
cs['vz'] = vz
cs['V_R'] = V_R
cs['V_phi'] = V_phi
cs['V_phi_rel'] = V_phi_rel
cs['V_Z'] = V_Z
cs['V_perp'] = V_perp
cs['V_tot'] = v_tot
cs['eccentricity'] = eccentricity
cs['Energy'] = Energy
cs['Lz'] = Lz
cs['energy_drift_pct'] = energy_drift_pct

#flag unstable orbit intergrations
DRIFT_THRESHOLD = 1.0  # percent

flagged_mask = np.abs(cs['energy_drift_pct']) > DRIFT_THRESHOLD
flagged = cs[flagged_mask].copy()

print(
    f"{len(flagged)}/{len(cs)} stars "
    f"({100 * len(flagged) / len(cs):.2f}%) "
    f"have |energy drift| > {DRIFT_THRESHOLD}%"
)

if len(flagged) > 0:
    print(flagged[['source_id', 'energy_drift_pct', 'Z_max']])

Processed 100/1721 stars
Processed 200/1721 stars
Processed 300/1721 stars
Processed 400/1721 stars
Processed 500/1721 stars
Processed 600/1721 stars
Processed 700/1721 stars
Processed 800/1721 stars
Processed 900/1721 stars
Processed 1000/1721 stars
Processed 1100/1721 stars
Processed 1200/1721 stars
Processed 1300/1721 stars
Processed 1400/1721 stars
Processed 1500/1721 stars
Processed 1600/1721 stars
Processed 1700/1721 stars
0/1721 stars (0.00%) have |energy drift| > 1.0%


In [67]:
cs.to_csv('../data/true_sources_final_kinematics.csv',index=False)